# Reusable Neural–DTB machinery: Cournot example

This Colab is intentionally small. The reusable implementation lives in two importable root-level files:

- `network.py`: MLP, identity-residual MLP, and hidden-block residual network choices.
- `dtb.py`: flat parameters, functional map evaluation, restricted parameter Jacobians, score-corrected velocity, SVD projection, score transport, and periodic map reset/refit.
- `test_dtb.py`: focused numerical and shape tests for the networks and DTB primitives.

The notebook only defines the Cournot drift, chooses settings, and runs the reusable driver.

## 1. Download the reusable files when running directly in Colab

When this notebook is run from a local clone, the files are imported in place. When opened directly from GitHub in Colab, this cell downloads the three small Python files from `main`.

In [ ]:
from pathlib import Path
import urllib.request

RAW_ROOT = "https://raw.githubusercontent.com/sun-mengwei/dtb-colab-experiments/refs/heads/codex/game-dynamics-dtb/DTB_Game_Resample_Repo"
for filename in ("network.py", "dtb.py", "test_dtb.py"):
    if not Path(filename).exists():
        print(f"Downloading {filename}")
        urllib.request.urlretrieve(f"{RAW_ROOT}/{filename}", filename)


## 2. Run the machinery tests

These tests independently check network output shapes, exact identity initialization, flatten/unflatten/write-back, `functional_call`, selected Jacobian values against float64 central finite differences, the diffusion-score term, exact SVD recovery on a known system, reset/refit error reduction, and a full score-aware DTB step.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "unittest", "-v", "test_dtb.py"],
    check=True,
)


## 3. Import the public API

The central objects are `ResidualMLP`, `ParticleState`, `NeuralDTB`, and `PeriodicReset`. Lower-level functions remain available when an experiment needs manual control.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.func import jacrev, vmap

from network import MLP, ResidualMLP, ResidualNetwork, count_trainable
from dtb import (
    NeuralDTB,
    PeriodicReset,
    dtb_basis_matrix,
    evaluate_map,
    flat_params,
    form_velocity,
    gaussian_particle_state,
    unflatten,
    write_flat_into_model,
)

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
if DEVICE.type == "cuda":
    torch.set_float32_matmul_precision("high")
print(f"PyTorch {torch.__version__} | device={DEVICE}")


## 4. Problem-specific code: multi-player Cournot drift

For player $i$, let $r_i=\sum_{j\ne i}x_j$. The equilibrium-consistent payoff and pseudo-gradient are

$$\Pi_i(x)=-b x_i^2+2b\mu x_i r_i(1-r_i),$$
$$b_i(x)=\partial_{x_i}\Pi_i=-2b x_i+2b\mu r_i-2b\mu r_i^2.$$

Only this drift function changes when the DTB machinery is reused for another game or dynamical system.

In [ ]:
def cournot_payoff(x: torch.Tensor, b: float = 1.0, mu: float = 2.0) -> torch.Tensor:
    """Return one equilibrium-consistent payoff per player."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    return -b * x.square() + 2.0 * b * mu * x * opponents * (1.0 - opponents)


def cournot_drift(x: torch.Tensor, b: float = 1.0, mu: float = 2.0) -> torch.Tensor:
    """Compute the simultaneous own-action payoff gradients."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    return -2.0 * b * x + 2.0 * b * mu * opponents - 2.0 * b * mu * opponents.square()


def own_action_gradient(payoff_fn, profiles: torch.Tensor) -> torch.Tensor:
    """Autodiff check: take the diagonal of each payoff Jacobian."""
    return vmap(lambda x: jacrev(payoff_fn)(x).diagonal())(profiles)


probe = torch.tensor([[0.2, 0.4], [0.5, 0.5]], device=DEVICE, dtype=DTYPE)
torch.testing.assert_close(own_action_gradient(cournot_payoff, probe), cournot_drift(probe))
print("Cournot own-action gradient check passed.")


## 5. Configure the neural tangent blocks

`last_layer_scale=1e-3` starts very close to the identity while keeping all MLP parameter derivatives nonzero. The network parameters and the selected coordinate basis remain frozen for `RESET_INTERVAL` physical steps. At a reset, the network is fit from the fixed labels $z_i$ to the current mapped particles $X_k(z_i)$, then a fresh flat parameter vector and tangent sub-basis are created.

The target velocity inside each step is $v=b(x)-\tfrac12Dq$, and `dtb.py` transports $x$, $\log\rho$, and $q=\nabla\log\rho$.

In [ ]:
FAST_MODE = True  # @param {type:"boolean"}
DIM = 2
N_PARTICLES = 256 if FAST_MODE else 1500
WIDTH = 16 if FAST_MODE else 64
DEPTH = 1 if FAST_MODE else 3
BASIS_SIZE = 64 if FAST_MODE else 512
STEPS = 30 if FAST_MODE else 100
STEP_SIZE = 0.01
RESET_INTERVAL = 10 if FAST_MODE else 20
RESET_OPTIMIZER_STEPS = 100 if FAST_MODE else 1000
SVD_RTOL = 1e-5
NOISE_STD = 0.10
INITIAL_STD = 0.12

model = ResidualMLP(
    dim=DIM, width=WIDTH, depth=DEPTH, activation="tanh",
    last_layer_scale=1e-3, dtype=DTYPE,
).to(DEVICE)
diffusion = NOISE_STD**2 * torch.eye(DIM, device=DEVICE, dtype=DTYPE)
initial_mean = torch.full((DIM,), 0.30, device=DEVICE, dtype=DTYPE)
cpu_generator = torch.Generator(device="cpu").manual_seed(SEED)
state = gaussian_particle_state(
    N_PARTICLES, initial_mean, INITIAL_STD, generator=cpu_generator
)

solver = NeuralDTB(
    model, cournot_drift, diffusion,
    step_size=STEP_SIZE,
    basis_size=BASIS_SIZE,
    svd_rtol=SVD_RTOL,
    jacobian_chunk_size=128,
    derivative_chunk_size=64,
    reset=PeriodicReset(
        interval=RESET_INTERVAL,
        optimizer_steps=RESET_OPTIMIZER_STEPS,
        learning_rate=3e-3,
        batch_size=128,
    ),
    seed=SEED,
)
print(f"trainable parameters={count_trainable(model):,} | selected basis={solver.selected.numel()}")


## 6. Inspect the flat parameters, functional map, and DTB basis

This is the low-level API used internally by `NeuralDTB`. It is included here so the tensor dimensions are visible before the experiment runs.

In [ ]:
theta_flat, structure = flat_params(model)
functional_values = evaluate_map(theta_flat, state.particles[:8], model, structure)
torch.testing.assert_close(functional_values, model(state.particles[:8]))

basis_preview = dtb_basis_matrix(
    theta_flat, solver.selected, state.particles[:8], model, structure, chunk_size=8
)
print("theta_flat:", tuple(theta_flat.shape))
print("map values:", tuple(basis_preview.values.shape))
print("J tensor:", tuple(basis_preview.jacobian.shape))
print("stacked J matrix:", tuple(basis_preview.matrix.shape))
print("first trainable tensors:", list(unflatten(theta_flat, structure))[:3])


## 7. Run the experiment

The loop is deliberately plain: one call advances the particle/density/score state, performs the SVD projection, and triggers a reset when scheduled.

In [ ]:
initial_state = state
snapshots = {0: state.particles.detach().cpu().clone()}
projection_residuals = []
retained_ranks = []
mean_score_norms = []
reset_steps, reset_before, reset_after = [], [], []
snapshot_steps = {STEPS // 2, STEPS}

start = time.perf_counter()
for step in range(1, STEPS + 1):
    result = solver.step(state, completed_steps=step)
    state = result.state
    projection_residuals.append(result.projection.relative_residual)
    retained_ranks.append(result.projection.rank)
    mean_score_norms.append(float(torch.linalg.vector_norm(state.score, dim=1).mean()))

    if result.reset is not None:
        reset_steps.append(step)
        reset_before.append(result.reset.rmse_before)
        reset_after.append(result.reset.rmse_after)
        print(
            f"reset at step {step:3d}: map RMSE "
            f"{result.reset.rmse_before:.3e} -> {result.reset.rmse_after:.3e}"
        )
    if step in snapshot_steps:
        snapshots[step] = state.particles.detach().cpu().clone()

elapsed = time.perf_counter() - start
print(f"completed {STEPS} steps in {elapsed:.2f} seconds")
print(f"final projection residual={projection_residuals[-1]:.3e}")


## 8. Results and diagnostics

In [ ]:
# Particle snapshots.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.1), constrained_layout=True)
all_points = torch.cat(list(snapshots.values())).numpy()
low, high = float(all_points.min()) - 0.05, float(all_points.max()) + 0.05
for ax, step in zip(axes, sorted(snapshots)):
    points = snapshots[step].numpy()
    ax.scatter(points[:, 0], points[:, 1], s=11, alpha=0.5, edgecolors="none")
    ax.scatter([0.0], [0.0], marker="x", s=90, c="goldenrod", label="saddle")
    ax.scatter([0.5], [0.5], marker="*", s=120, c="crimson", label="stable")
    ax.set(
        title=f"t={step * STEP_SIZE:.2f}", xlabel="player 1 quantity",
        ylabel="player 2 quantity", xlim=(low, high), ylim=(low, high),
    )
    ax.grid(alpha=0.2)
axes[-1].legend()
plt.show()

# Projection, rank, score, and reset diagnostics.
step_axis = np.arange(1, STEPS + 1)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.7), constrained_layout=True)
axes[0].semilogy(step_axis, np.maximum(projection_residuals, 1e-12))
axes[0].set(title="Tangent projection", xlabel="step", ylabel="relative residual")
axes[1].plot(step_axis, retained_ranks)
axes[1].set(title="Retained SVD rank", xlabel="step", ylabel="rank")
axes[2].plot(step_axis, mean_score_norms)
axes[2].set(title="Transported score", xlabel="step", ylabel="mean ||q||")
if reset_steps:
    axes[3].semilogy(reset_steps, reset_before, "o--", label="before")
    axes[3].semilogy(reset_steps, reset_after, "o-", label="after")
    axes[3].legend()
axes[3].set(title="Periodic map reset", xlabel="step", ylabel="map RMSE")
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()


## Reusing the machinery

For another setting, import `network.py` and `dtb.py`, replace `cournot_drift`, choose a square diffusion matrix, and provide an initial `ParticleState` (or use `gaussian_particle_state`). Use `MLP` when no identity skip is wanted, `ResidualMLP` for pushforward maps, or `ResidualNetwork` for deeper hidden skip connections. Set `PeriodicReset(interval=0)` to disable refits, or adjust its optimizer controls independently of the physical time step.